In [1]:
import pandas as pd
import numpy as np

inventory = pd.read_csv(
    "../exports/cleaned_data/inventory_clean.csv"
)

inventory_new = pd.read_csv(
    "../exports/cleaned_data/inventory_new_clean.csv"
)

products = pd.read_csv(
    "../exports/cleaned_data/products_clean.csv"
)

product_features = pd.read_csv(
    "../exports/processed_data/product_features.csv"
)

In [2]:
inventory_master = (
    inventory.merge(
        products,
        on="product_id",
        how="left"
    )
)

inventory_master.head()

,product_id,date,stock_received,damaged_stock,product_name,category,brand,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level
0,153019,17-03-2023,4,2,Onions,Fruits & Vegetables,Aurora LLC,947.95,1263.93,25.0,3,13,88
1,848226,17-03-2023,4,2,Tomatoes,Fruits & Vegetables,Barad and Sons,209.59,279.45,25.0,3,10,51
2,965755,17-03-2023,1,0,Bananas,Fruits & Vegetables,"Doshi, Sarraf and Sama",532.57,710.09,25.0,3,21,91
3,39154,17-03-2023,4,0,Mangoes,Fruits & Vegetables,"Suresh, Bose and Bajwa",946.86,1262.48,25.0,3,21,84
4,34186,17-03-2023,3,2,Mangoes,Fruits & Vegetables,Mandal-Kar,925.65,1234.20,25.0,3,27,74


In [18]:
inventory_master["damage_rate"] = np.where(
    inventory_master["stock_received"] > 0,
    inventory_master["damaged_stock"]
    /
    inventory_master["stock_received"],
    0
)

In [19]:
inventory_master["damage_rate"] = (
    inventory_master["damage_rate"]
    .fillna(0)
)

In [20]:
inventory_master["inventory_health_score"] = np.where(
    inventory_master["stock_received"] > 0,

    (
        (
            inventory_master["stock_received"]
            -
            inventory_master["damaged_stock"]
        )
        /
        inventory_master["stock_received"]
    ) * 100,

    0
)

In [21]:
inventory_master[
    "inventory_health_score"
] = (
    inventory_master[
        "inventory_health_score"
    ]
    .fillna(100)
)

In [22]:
category_inventory_health = (
    inventory_master.groupby(
        "category"
    )
    .agg({
        "stock_received":"sum",
        "damaged_stock":"sum",
        "inventory_health_score":"mean"
    })
    .reset_index()
)

category_inventory_health

,category,stock_received,damaged_stock,inventory_health_score
0,Baby Care,9700,4828,54.990249
1,Cold Drinks & Juices,11107,6604,48.160744
2,Dairy & Breakfast,14873,8890,47.922736
3,Fruits & Vegetables,13670,8120,47.922450
4,Grocery & Staples,15036,7144,54.102992
5,Household Care,12034,8210,45.332781
6,Instant & Frozen Food,10183,5910,48.959509
7,Personal Care,12239,7524,48.439992
8,Pet Care,17521,7528,58.340735
9,Pharmacy,17318,7366,58.293849


In [23]:
inventory_master["reorder_flag"] = np.where(
    inventory_master["stock_received"]
    <
    inventory_master["min_stock_level"],
    "Reorder Needed",
    "Stock Sufficient"
)

In [24]:
inventory_master[
    "reorder_flag"
].value_counts()

reorder_flag
Reorder Needed    75172
Name: count, dtype: int64

In [25]:
inventory_master["risk_score"] = (
    inventory_master["damage_rate"]
    * 100
)

In [26]:
inventory_master["risk_category"] = pd.cut(
    inventory_master["risk_score"],
    bins=[-1,5,15,100],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

In [27]:
product_inventory = (
    product_features.merge(
        inventory_master,
        on="product_id",
        how="left"
    )
)

In [28]:
product_inventory["performance_score"] = (
    product_inventory["revenue"]
    *
    product_inventory[
        "inventory_health_score"
    ]
)

In [29]:
top_risk_products = (
    inventory_master
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(20)
)

top_risk_products

,product_id,date,stock_received,damaged_stock,product_name,category,brand,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level,damage_rate,inventory_health_score,reorder_flag,risk_score,risk_category
33,602241,17-03-2023,1,2,Nuts,Snacks & Munchies,Bahl-Pau,976.55,1502.38,35.0,90,12,75,2.0,-100.0,Reorder Needed,200.0,NaN
30402,764014,09-11-2023,1,2,Salt,Grocery & Staples,"Grewal, Swamy and Dayal",612.04,720.05,15.0,365,30,100,2.0,-100.0,Reorder Needed,200.0,NaN
30392,470449,09-11-2023,1,2,Frozen Biryani,Instant & Frozen Food,Kohli-Sawhney,827.77,1379.62,40.0,180,27,57,2.0,-100.0,Reorder Needed,200.0,NaN
30211,473929,08-11-2023,1,2,Carrots,Fruits & Vegetables,"Bhardwaj, Bhatia and Deshmukh",440.19,586.92,25.0,3,19,70,2.0,-100.0,Reorder Needed,200.0,NaN
30186,564692,07-11-2023,1,2,Cat Food,Pet Care,"Krishna, Bail and Dyal",215.65,331.77,35.0,365,25,82,2.0,-100.0,Reorder Needed,200.0,NaN
11922,901144,17-06-2023,1,2,Sugar,Grocery & Staples,Deo-Kamdar,340.36,400.42,15.0,365,12,71,2.0,-100.0,Reorder Needed,200.0,NaN
11793,451933,16-06-2023,1,2,Wheat Flour,Grocery & Staples,Nadig Ltd,568.94,669.34,15.0,365,28,98,2.0,-100.0,Reorder Needed,200.0,NaN
11790,327290,16-06-2023,1,2,Sugar,Grocery & Staples,Nadkarni Inc,862.28,1014.45,15.0,365,22,65,2.0,-100.0,Reorder Needed,200.0,NaN
11786,945635,16-06-2023,1,2,Pulses,Grocery & Staples,Balasubramanian PLC,778.90,916.35,15.0,365,16,59,2.0,-100.0,Reorder Needed,200.0,NaN
30322,706112,08-11-2023,1,2,Vitamins,Pharmacy,"Magar, Gara and Garg",478.79,598.49,20.0,365,12,77,2.0,-100.0,Reorder Needed,200.0,NaN


In [30]:
category_inventory_health.sort_values(
    "damaged_stock",
    ascending=False
)

,category,stock_received,damaged_stock,inventory_health_score
2,Dairy & Breakfast,14873,8890,47.922736
5,Household Care,12034,8210,45.332781
10,Snacks & Munchies,13845,8144,48.459422
3,Fruits & Vegetables,13670,8120,47.922450
8,Pet Care,17521,7528,58.340735
7,Personal Care,12239,7524,48.439992
9,Pharmacy,17318,7366,58.293849
4,Grocery & Staples,15036,7144,54.102992
1,Cold Drinks & Juices,11107,6604,48.160744
6,Instant & Frozen Food,10183,5910,48.959509


In [31]:
inventory_master["stock_gap"] = (
    inventory_master["min_stock_level"]
    -
    inventory_master["stock_received"]
)

In [32]:
inventory_master["reorder_risk"] = np.where(
    inventory_master["stock_received"]
    <
    inventory_master["min_stock_level"],
    1,
    0
)

In [33]:
category_inventory_health

,category,stock_received,damaged_stock,inventory_health_score
0,Baby Care,9700,4828,54.990249
1,Cold Drinks & Juices,11107,6604,48.160744
2,Dairy & Breakfast,14873,8890,47.922736
3,Fruits & Vegetables,13670,8120,47.922450
4,Grocery & Staples,15036,7144,54.102992
5,Household Care,12034,8210,45.332781
6,Instant & Frozen Food,10183,5910,48.959509
7,Personal Care,12239,7524,48.439992
8,Pet Care,17521,7528,58.340735
9,Pharmacy,17318,7366,58.293849


In [34]:
inventory_master["risk_category"].value_counts()

risk_category
Low Risk       63449
High Risk      10879
Medium Risk        0
Name: count, dtype: int64

In [35]:
inventory_master["risk_score"].describe()

count    75172.000000
mean        11.130918
std         29.655405
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        200.000000
Name: risk_score, dtype: float64

In [36]:
inventory_master["risk_score"].quantile(
    [0.25,0.50,0.75]
)

0.25    0.0
0.50    0.0
0.75    0.0
Name: risk_score, dtype: float64

In [38]:
inventory_master["risk_score"].value_counts().head(20)

risk_score
0.000000      63449
66.666667      7439
50.000000      3440
200.000000      844
Name: count, dtype: int64

In [39]:
inventory_master["risk_category"] = pd.cut(
    inventory_master["risk_score"],
    bins=[-1, 10, 70, 1000],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

In [40]:
inventory_master["risk_category"].value_counts()

risk_category
Low Risk       63449
Medium Risk    10879
High Risk        844
Name: count, dtype: int64

In [41]:
inventory_master["risk_category"] = pd.cut(
    inventory_master["risk_score"],
    bins=[-1,10,70,1000],
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)

In [42]:
category_inventory_health[
    "damage_percentage"
] = (
    category_inventory_health[
        "damaged_stock"
    ]
    /
    category_inventory_health[
        "stock_received"
    ]
) * 100

In [43]:
top_risk_products = (
    inventory_master
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(50)
)

In [44]:
category_risk_summary = (
    inventory_master
    .groupby("category")
    .agg({
        "risk_score":"mean",
        "damage_rate":"mean",
        "inventory_health_score":"mean"
    })
    .reset_index()
)

In [45]:
print("""
Supply Chain Intelligence Insights

• Pet Care and Pharmacy show the strongest inventory health.

• Household Care exhibits the weakest inventory health.

• 844 products fall into the High Risk category.

• 10,879 products fall into the Medium Risk category.

• Inventory optimization efforts should prioritize high-risk products and low-health categories.

• Damage rates indicate operational inefficiencies in selected categories.
""")


Supply Chain Intelligence Insights

• Pet Care and Pharmacy show the strongest inventory health.

• Household Care exhibits the weakest inventory health.

• 844 products fall into the High Risk category.

• 10,879 products fall into the Medium Risk category.

• Inventory optimization efforts should prioritize high-risk products and low-health categories.

• Damage rates indicate operational inefficiencies in selected categories.



In [46]:
output_path = "../exports/supply_chain_output/"

In [47]:
inventory_master.to_csv(
    output_path + "inventory_master.csv",
    index=False
)

category_inventory_health.to_csv(
    output_path + "category_inventory_health.csv",
    index=False
)

category_risk_summary.to_csv(
    output_path + "category_risk_summary.csv",
    index=False
)

product_inventory.to_csv(
    output_path + "product_inventory_matrix.csv",
    index=False
)

top_risk_products.to_csv(
    output_path + "top_risk_products.csv",
    index=False
)